# 04c - Demo Fitur Polinomial dan Regularisasi
Materi Modul 4: lab demo + demo bagian 1-3 (fitur polinomial dan regularisasi).

Fitur polinomial membuat jumlah fitur meledak -> risiko overfitting tinggi.
Regularisasi dipakai untuk menjinakkannya. Alur lengkap: **Polynomial -> Scaler -> Model berpenalti**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error

df = pd.read_csv('../datasets/diabetes.csv')
X = df.drop(columns='progres_penyakit')
y = df['progres_penyakit']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 1. Fitur polinomial derajat 2 (10 fitur -> 65 fitur)

In [ ]:
pf = PolynomialFeatures(degree=2, include_bias=False)
X_poly = pf.fit_transform(X_train)
print('Sebelum:', X_train.shape[1], 'fitur | Sesudah:', X_poly.shape[1], 'fitur')
print(pf.get_feature_names_out()[:15])

## 2. Tanpa regularisasi vs dengan regularisasi

In [ ]:
def evaluasi(nama, model):
    model.fit(X_train, y_train)
    r_tr = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    r_te = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    print(f'{nama:32s} | RMSE train={r_tr:6.2f} | RMSE test={r_te:6.2f}')

pf2 = lambda: PolynomialFeatures(degree=2, include_bias=False)

evaluasi('Linear derajat 1', make_pipeline(StandardScaler(), LinearRegression()))
evaluasi('Linear derajat 2 (overfit)', make_pipeline(pf2(), StandardScaler(), LinearRegression()))
evaluasi('Ridge derajat 2 (alpha=100)', make_pipeline(pf2(), StandardScaler(), Ridge(alpha=100)))
evaluasi('Lasso derajat 2 (alpha=2)', make_pipeline(pf2(), StandardScaler(), Lasso(alpha=2, max_iter=20000)))

## 3. Cari alpha terbaik dengan `GridSearchCV`

In [ ]:
pipe = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), StandardScaler(), Ridge())
param = {'ridge__alpha': np.logspace(-1, 4, 20)}

grid = GridSearchCV(pipe, param, cv=KFold(5, shuffle=True, random_state=42), scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)

print('Alpha terbaik   :', round(grid.best_params_['ridge__alpha'], 2))
print('RMSE CV terbaik :', round(np.sqrt(-grid.best_score_), 2))
print('RMSE test       :', round(np.sqrt(mean_squared_error(y_test, grid.predict(X_test))), 2))

## 4. Visualisasi skor CV terhadap alpha

In [ ]:
hasil = pd.DataFrame(grid.cv_results_)
plt.semilogx(hasil['param_ridge__alpha'].astype(float), np.sqrt(-hasil['mean_test_score']), marker='o')
plt.xlabel('alpha'); plt.ylabel('RMSE (CV)'); plt.title('Memilih alpha Ridge pada fitur polinomial')
plt.show()

## 5. Berapa fitur yang selamat di Lasso?

In [ ]:
lasso_poly = make_pipeline(pf2(), StandardScaler(), Lasso(alpha=2, max_iter=20000)).fit(X_train, y_train)
nama_fitur = lasso_poly[0].get_feature_names_out(X.columns)
koef = pd.Series(lasso_poly[-1].coef_, index=nama_fitur)
print('Fitur total      :', len(koef))
print('Fitur koefisien 0:', (koef == 0).sum())
print('Fitur dipakai    :', (koef != 0).sum())
koef[koef != 0].sort_values(key=abs, ascending=False).round(2)

## Catatan
- (urutan pipeline kenapa Polynomial dulu baru Scaler, efek regularisasi pada model polinomial, dll.)